In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # dimension is 384

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import pandas as pd

df = pd.read_csv("cleaned_police_reports.csv")

In [ ]:
df.head(3)

,name,department,url,text
0,"Andrew Allen, badge #37",Minneapolis Police Department,https://d3n8a8pro7vhmx.cloudfront.net/cuapb/pa...,Home Legislative File 2021-01132 RCA Legal s...
1,"Guled Abdullahi, badge #706",Hennepin County Sheriff's Department,https://assets.nationbuilder.com/cuapb/pages/1...,"Hennepin County 300 South Sixth Street, Minne..."
2,"Dean V. Albers, badge #None",Goodhue County Sheriff's Department,https://d3n8a8pro7vhmx.cloudfront.net/cuapb/pa...,"2/22/2021 Jenson v. Craft, Civil No. 01-1488(D..."


In [ ]:
vectors = model.encode(df, convert_to_numpy=True)

In [ ]:
!pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0


In [ ]:
from pinecone import Pinecone
import os
#api key goes here
pc = Pinecone(api_key=PINECONE)
index_name = "cuapb"

index = pc.Index(index_name)

In [ ]:
import math

def chunk_text(text, chunk_size=400, overlap=50):
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for i in range(0, max(1, len(words)), step):
        chunk = " ".join(words[i:i+chunk_size])
        if not chunk:
            break
        start_word = i
        end_word = i + len(chunk.split()) - 1
        chunks.append((chunk, start_word, end_word))
        if len(words) <= i + chunk_size:
            break
    return chunks

In [ ]:

items = []
for row_idx, row in df.iterrows():
    doc_id = f"r{row_idx}"
    name = row["name"]
    department = row["department"]
    url = row["url"]
    text = row["text"]
    chunks = chunk_text(text, chunk_size=300, overlap=50)
    for chunk_idx, (chunk, start, end) in enumerate(chunks):
        excerpt = chunk[:300]
        metadata = {
            "name": name,
            "department": department,
             "url": url,
             "start": start,
             "end": end,
            "excerpt": excerpt[:250]
        }

        items.append({
            "doc_id": doc_id,
            "row_idx": row_idx,
            "chunk_idx": chunk_idx,
            "text": f"{name} {department} {chunk}",
            "metadata": metadata
        })


In [ ]:
payloads=[]
for i in items:
  encoding=model.encode(i["text"], batch_size=32, convert_to_numpy=True)

  # Replace NaN values with None
  metadata = {
      "name": i["metadata"]["name"],
      "department": None if pd.isna(i["metadata"]["department"]) else i["metadata"]["department"],
      "url": i["metadata"]["url"],
      "start": i["metadata"]["start"],
      "end": i["metadata"]["end"],
      "excerpt": i["metadata"]["excerpt"]
  }

  payloads.append({
            "id": f"{i['doc_id']}-{i['chunk_idx']}",
            "values": encoding.tolist(),
            "metadata": metadata
        })

In [ ]:
MAX_BATCH_SIZE = 100  # tune based on vector size
for i in range(0, len(payloads), MAX_BATCH_SIZE):
    batch = payloads[i:i + MAX_BATCH_SIZE]
    index.upsert(vectors=batch)


PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Wed, 05 Nov 2025 04:09:05 GMT', 'Content-Type': 'application/json', 'Content-Length': '138', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '13', 'x-pinecone-request-id': '5898161979983432497', 'x-envoy-upstream-service-time': '3', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Metadata value must be a string, number, boolean or list of strings, got 'null' for field 'department'","details":[]}


In [ ]:
def sanitize_metadata(meta):
    return {
        k: v if v is not None else ""  # or remove the key entirely
        for k, v in meta.items()
        if v is None or isinstance(v, (str, int, float, bool, list))
    }

cleaned_payloads = []
for item in payloads:
    item["metadata"] = sanitize_metadata(item.get("metadata", {}))
    cleaned_payloads.append(item)

# Then batch and upsert
for i in range(0, len(cleaned_payloads), MAX_BATCH_SIZE):
    batch = cleaned_payloads[i:i + MAX_BATCH_SIZE]
    index.upsert(vectors=batch)

In [ ]:
index.upsert(vectors=payloads)

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Wed, 05 Nov 2025 04:07:03 GMT', 'Content-Type': 'application/json', 'Content-Length': '119', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '3797', 'x-pinecone-request-id': '754367655336819917', 'x-envoy-upstream-service-time': '1', 'server': 'envoy'})
HTTP response body: {"code":11,"message":"Error, message length too large: found 50620972 bytes, the limit is: 4194304 bytes","details":[]}


In [ ]:
q_vec = model.encode("Abusive language", convert_to_numpy=True).tolist()
res = index.query(vector=q_vec, top_k=10, include_metadata=True, include_values=False)

In [ ]:
res["matches"]

[{'id': 'r361-11',
  'metadata': {'department': 'St. Paul Police Department',
               'end': 3049.0,
               'excerpt': 'obscene, abusive, boisterous, or noisy conduct or in '
                          'offensive, obscene, or abusive language tending '
                          'reasonably to arouse alarm, anger, or resentment in '
                          'others,” if the person engaging in such conduct '
                          '“know[s], or ha[s] reasonable grounds to know tha',
               'name': 'Gerald A. Carter, badge #98600',
               'start': 2750.0,
               'url': 'https://d3n8a8pro7vhmx.cloudfront.net/cuapb/pages/270/attachments/original/1625841558/maeberry_USCOURTS-mnd-0_09-cv-01216-0.pdf?1625841558'},
  'score': 0.554867268,
  'values': []},
 {'id': 'r1821-0',
  'metadata': {'department': 'Minneapolis Police Department',
               'end': 299.0,
               'excerpt': 'prohibition against using derogatory, indecent, '
              